**Install and Load**

In [ ]:
!pip install -q datasets emoji langdetect

from datasets import load_dataset
import pandas as pd

dataset = load_dataset("QCRI/HumAID-all",verification_mode="no_checks")

df = dataset["train"].to_pandas()

df.head()

**Check columns**

In [ ]:
df.columns

**Dataset Shape**

In [ ]:
df.shape

**Basic Information**

In [ ]:
df.info()

**Missing Values**

In [ ]:
df.isnull().sum()

**Class Distribution**

In [ ]:
df["class_label"].value_counts()

In [ ]:
df["class_label"].value_counts().plot(
    kind="bar",
    title="Class Distribution",
    xlabel="Class",
    ylabel="Number of Tweets"
)

**Class Distribution Percentage**

In [ ]:
df["class_label"].value_counts(normalize=True) * 100

In [ ]:
df["class_label"].value_counts(normalize=True).plot(
    kind="pie",
    title="Class Distribution Percentage",
    autopct="%1.1f%%"
)

**Duplicate Tweets**

In [ ]:
df["tweet_text"].duplicated().sum()

**Duplicate character Length**

In [ ]:
df["char_length"] = df["tweet_text"].str.len()


In [ ]:
df["char_length"].describe()

In [ ]:
df["char_length"].plot(kind="hist",bins=50, title="Distribution of Tweet Chracter Length", xlabel="Number of characters",ylabel="frequency")

**Tweet Word Count**

In [ ]:
df["word_count"] = df["tweet_text"].str.split().str.len()

In [ ]:
df["word_count"].describe()

In [ ]:
df["word_count"].plot(
    kind="hist",
    bins=30,
    title="Distribution of Tweet Word Count",
    xlabel="Number of Words",
    ylabel="Frequency"
)

**Top 20 Most Frequent Words**

In [ ]:
from collections import Counter

word_counts = Counter(
    " ".join(df["tweet_text"].dropna()).lower().split()
)

top_words = pd.DataFrame(
    word_counts.most_common(20),
    columns=["Word", "Frequency"]
)

top_words

In [ ]:
top_words.plot(
    x="Word",
    y="Frequency",
    kind="barh",
    title="Top 20 Most Frequent Words",
    xlabel="Frequency",
    ylabel="Word"
)

**Top 20 Bigams**

In [ ]:
from collections import Counter

bigrams = []

for text in df["tweet_text"].dropna().str.lower():
    words = text.split()

    for i in range(len(words) - 1):
        bigrams.append(words[i] + " " + words[i + 1])

bigram_counts = Counter(bigrams)

top_bigrams = pd.DataFrame(
    bigram_counts.most_common(20),
    columns=["Bigram", "Frequency"]
)

top_bigrams

In [ ]:
top_bigrams.plot(
    x="Bigram",
    y="Frequency",
    kind="barh",
    title="Top 20 Most Frequent Bigrams",
    xlabel="Frequency",
    ylabel="Bigram"
)

**Top Hashtags**

In [ ]:
import re

hashtags = []

for text in df["tweet_text"].dropna():
    hashtags.extend(re.findall(r"#\w+", text.lower()))

hashtag_counts = Counter(hashtags)

top_hashtags = pd.DataFrame(
    hashtag_counts.most_common(20),
    columns=["Hashtag", "Frequency"]
)

top_hashtags

In [ ]:
top_hashtags.plot(
    x="Hashtag",
    y="Frequency",
    kind="barh",
    title="Top 20 Hashtags",
    xlabel="Frequency",
    ylabel="Hashtag"
)

**Mentions**

In [ ]:
mentions = []

for text in df["tweet_text"].dropna():
    mentions.extend(re.findall(r"@\w+", text))

Counter(mentions).most_common(20)

**URLs**

In [ ]:
url_count = df["tweet_text"].str.contains(
    r"http\S+|www\S+",
    case=False,
    regex=True
).value_counts()

url_count

In [ ]:
url_count.plot(
    kind="bar",
    title="Tweets With and Without URLs",
    xlabel="Contains URL",
    ylabel="Number of Tweets"
)

**Sample Tweets**

In [ ]:
df[["tweet_text", "class_label"]].sample(10)

In [ ]:
df.groupby("class_label")["tweet_text"].apply(
    lambda x: x.sample(min(3, len(x)))
)

**Raw vs Cleaned Text**

In [ ]:
import re

def clean_text(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove mentions
    text = re.sub(r"@\w+", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
df["cleaned_text"] = df["tweet_text"].apply(clean_text)

In [ ]:
df[["tweet_text", "cleaned_text"]].head()

In [ ]:
df.head()

In [ ]:
df.to_csv("HumAID_train_EDA.csv", index=False)

In [ ]:
from google.colab import files

files.download("HumAID_train_EDA.csv")

**NLP Dataset**

In [ ]:
import os

if not os.path.exists("train.csv"):
    from google.colab import files
    files.upload()

In [ ]:
import pandas as pd

data = pd.read_csv("train.csv")

data.head()

In [ ]:
data.shape

In [ ]:
data.isnull().sum()

**Exact duplicates**

In [ ]:
data["text"].duplicated().sum()

In [ ]:
data[data["text"].duplicated(keep=False)].sort_values("text")

In [ ]:
data[data["text"].duplicated(keep=False)][
    ["text", "target"]
].sort_values("text")

In [ ]:
data.shape

In [ ]:
data = data.drop_duplicates(subset="text", keep="first")

In [ ]:
print("Rows after removing duplicates:", len(data))
print("Duplicate tweets left:", data["text"].duplicated().sum())

**Normalized duplicates**

In [ ]:
import re
import html

def normalize_text(text):
    text = html.unescape(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^\w\s#]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

data["normalized_text"] = data["text"].apply(normalize_text)

data["normalized_text"].duplicated().sum()

**Class Distribution**

In [ ]:
data["target"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

data["target"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(7,5)
)

plt.title("Class Distribution")
plt.xlabel("Target")
plt.ylabel("Number of Tweets")
plt.xticks([0, 1], ["Not Disaster", "Disaster"], rotation=0)

plt.show()

In [ ]:
data["target"].value_counts().plot(
    kind="pie",
    autopct="%1.1f%%",
    figsize=(6,6)
)

plt.title("Class Distribution Percentage")
plt.ylabel("")

plt.show()

**Character length**

In [ ]:
data["char_length"] = data["text"].apply(len)

data["char_length"].describe()

In [ ]:
plt.figure(figsize=(10,6))

plt.hist(data["char_length"], bins=30)

plt.title("Tweet Character Length")
plt.xlabel("Number of Characters")
plt.ylabel("Number of Tweets")

plt.show()

**Word count**

In [ ]:
data["word_count"] = data["text"].apply(
    lambda x: len(x.split())
)

data["word_count"].describe()

In [ ]:
plt.figure(figsize=(10,6))

plt.hist(data["word_count"], bins=30)

plt.title("Tweet Word Count")
plt.xlabel("Number of Words")
plt.ylabel("Number of Tweets")

plt.show()

**Most common words**

In [ ]:
from collections import Counter

words = []

for tweet in data["text"]:

    tweet = normalize_text(tweet)

    words.extend(tweet.split())

word_counts = Counter(words)

word_counts.most_common(20)

**Top 20 words**

In [ ]:
top_words = word_counts.most_common(20)

words = [x[0] for x in top_words]
counts = [x[1] for x in top_words]

plt.figure(figsize=(10,7))

plt.barh(words[::-1], counts[::-1])

plt.title("Top 20 Most Common Words")
plt.xlabel("Frequency")
plt.ylabel("Word")

plt.show()

**Bigrams**

In [ ]:
from collections import Counter

bigram_counts = Counter()

for tweet in data["text"]:

    words = normalize_text(tweet).split()

    for i in range(len(words) - 1):

        bigram = words[i] + " " + words[i+1]

        bigram_counts[bigram] += 1

bigram_counts.most_common(20)

In [ ]:
top_bigrams = bigram_counts.most_common(20)

bigrams = [x[0] for x in top_bigrams]
counts = [x[1] for x in top_bigrams]

plt.figure(figsize=(10,7))

plt.barh(bigrams[::-1], counts[::-1])

plt.title("Top 20 Most Common Bigrams")
plt.xlabel("Frequency")
plt.ylabel("Bigram")

plt.show()

**Hashtags**

In [ ]:
hashtags = []

for tweet in data["text"]:

    hashtags.extend(
        re.findall(r"#\w+", tweet.lower())
    )

hashtag_counts = Counter(hashtags)

hashtag_counts.most_common(20)

In [ ]:
top_hashtags = hashtag_counts.most_common(20)

hashtags = [x[0] for x in top_hashtags]
counts = [x[1] for x in top_hashtags]

plt.figure(figsize=(10,7))

plt.barh(hashtags[::-1], counts[::-1])

plt.title("Top 20 Hashtags")
plt.xlabel("Frequency")
plt.ylabel("Hashtag")

plt.show()

**Mentions**

In [ ]:
mentions = []

for tweet in data["text"]:

    mentions.extend(
        re.findall(r"@\w+", tweet)
    )

mention_counts = Counter(mentions)

print("Total mentions:", len(mentions))
print("Top mentions:", mention_counts.most_common(10))

**Tweets containing URLs**

In [ ]:
data["has_url"] = data["text"].apply(
    lambda x: bool(
        re.search(r"http\S+|www\S+", x)
    )
)

data["has_url"].value_counts()

In [ ]:
data["has_url"].value_counts().plot(
    kind="bar",
    figsize=(7,5)
)

plt.title("Tweets Containing URLs")
plt.xlabel("Contains URL")
plt.ylabel("Number of Tweets")
plt.xticks(
    [0, 1],
    ["No URL", "Contains URL"],
    rotation=0
)

plt.show()

**Emojis**

In [ ]:
import emoji

emoji_list = []

for tweet in data["text"]:

    emoji_list.extend(
        [x["emoji"] for x in emoji.emoji_list(tweet)]
    )

emoji_counts = Counter(emoji_list)

print("Total emojis:", len(emoji_list))
print("Unique emojis:", len(emoji_counts))
print("Top emojis:", emoji_counts.most_common(10))

**Random tweets**

In [ ]:
data.sample(10)[
    ["text", "keyword", "location", "target"]
]

**Raw vs Cleaned text**

In [ ]:
def clean_text(text):

    text = html.unescape(text)

    text = text.lower()

    text = re.sub(
        r"http\S+|www\S+",
        "",
        text
    )

    text = re.sub(
        r"@\w+",
        "",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


data["cleaned_text"] = data["text"].apply(clean_text)

In [ ]:
data[["text", "cleaned_text"]].sample(5)

**Download Dataset**

In [ ]:
data.to_csv("train.csv",index=False)

**CrisisLexT26 Dataset**

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import re
import html
import string

from collections import Counter

**Find the 26 CSV files**

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
import zipfile
import os

zip_path = "/content/CrisisLexT26-v1.0.zip"
extract_path = "/content/CrisisLexT26_extracted"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed!")
print("Extracted to:", extract_path)

In [ ]:
import os

labeled_files = []

for root, dirs, files in os.walk("/content/CrisisLexT26_extracted"):
    for file in files:
        if file.endswith("-tweets_labeled.csv"):
            labeled_files.append(os.path.join(root, file))

print("Labeled CSV files:", len(labeled_files))

for file in labeled_files:
    print(file)

In [ ]:
Crisis_dfs = []

for file in labeled_files:

    crisis_df = pd.read_csv(file)

    # Store original filename
    crisis_df["source_file"] = os.path.basename(file)

    Crisis_dfs.append(crisis_df)

print("Files loaded:", len(Crisis_dfs))

Check each file's size

In [ ]:
for i, crisis_df in enumerate(Crisis_dfs):

    print(
        i + 1,
        "|",
        crisis_df["source_file"].iloc[0],
        "| Rows:",
        len(crisis_df),
        "| Columns:",
        len(crisis_df.columns)
    )

**Check schema across all files**

In [ ]:
for i, crisis_df in enumerate(Crisis_dfs):

    print(f"\nFile {i+1}:")
    print(crisis_df.columns.tolist())

**Automatically check whether schemas match**

In [ ]:
reference_columns = list(Crisis_dfs[0].columns)

print("Reference columns:")
print(reference_columns)

print("\nSchema check:\n")

for i, crisis_df in enumerate(Crisis_dfs):

    current_columns = list(crisis_df.columns)

    if current_columns == reference_columns:

        print(f"{i+1}. PASS")

    else:

        print(f"{i+1}. DIFFERENT")

        print("Missing:",
              set(reference_columns) - set(current_columns))

        print("Extra:",
              set(current_columns) - set(reference_columns))

**Combine all 26 files**

In [ ]:
CrisisLex26 = pd.concat(
    Crisis_dfs,
    ignore_index=True
)

print("Total rows:", len(CrisisLex26))
print("Total columns:", len(CrisisLex26.columns))

In [ ]:
CrisisLex26.head()

**Check column names**

In [ ]:
print(CrisisLex26.columns.tolist())

In [ ]:
CrisisLex26.columns = CrisisLex26.columns.str.strip()

print(CrisisLex26.columns.tolist())

**Identify the text column**

In [ ]:
text_col = "Tweet Text"

print("Text column:", text_col)
print(CrisisLex26[text_col].head())

**Universal Checks**

In [ ]:
len(CrisisLex26)

Missing/null values in Tweet Tecxt

In [ ]:
missing_text = CrisisLex26[text_col].isna().sum()

empty_text = (
    CrisisLex26[text_col]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Missing Tweet Text:", missing_text)
print("Empty Tweet Text:", empty_text)

Exact Duplicate rows

In [ ]:
duplicate_rows = CrisisLex26.duplicated().sum()

print("Exact duplicate rows:", duplicate_rows)

Duplicate Tweet Text

In [ ]:
duplicate_text = (
    CrisisLex26[text_col]
    .fillna("")
    .astype(str)
    .duplicated()
    .sum()
)

print("Duplicate tweet texts:", duplicate_text)

In [ ]:
duplicate_texts = CrisisLex26[CrisisLex26["Tweet Text"].duplicated(keep=False)]

print("Total duplicate tweet rows:", len(duplicate_texts))
print("Unique duplicate texts:", duplicate_texts["Tweet Text"].nunique())

In [ ]:
duplicate_texts[["Tweet Text"]].sort_values("Tweet Text").head(20)

In [ ]:
print("Total tweets:", len(CrisisLex26))
print("Duplicate tweet texts:", CrisisLex26["Tweet Text"].duplicated().sum())
print("Unique tweet texts:", CrisisLex26["Tweet Text"].nunique())

In [ ]:
duplicate_percentage = (
    CrisisLex26["Tweet Text"].duplicated().sum()
    / len(CrisisLex26)
) * 100

print(f"Duplicate tweet percentage: {duplicate_percentage:.2f}%")

**Near-duplicate/retweet check**

In [ ]:
def normalize_for_duplicate_check(text):

    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove @mentions
    text = re.sub(r"@\w+", "", text)

    # Remove RT
    text = re.sub(r"\brt\b", "", text)

    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
CrisisLex26["normalized_text_check"] = (
    CrisisLex26[text_col]
    .fillna("")
    .apply(normalize_for_duplicate_check)
)

near_duplicates = (
    CrisisLex26["normalized_text_check"]
    .duplicated()
    .sum()
)

print("Near-duplicate candidates:", near_duplicates)

**Check actual retweets**

In [ ]:
retweet_mask = (
    CrisisLex26[text_col]
    .fillna("")
    .astype(str)
    .str.match(
        r"^\s*RT\b",
        case=False
    )
)

print("Retweets:", retweet_mask.sum())

print(
    "Retweet percentage:",
    round(retweet_mask.mean() * 100, 2),
    "%"
)

In [ ]:
CrisisLex26.loc[
    retweet_mask,
    [text_col]
].head(20)

**Check Tweets becoming empty after cleaning**

In [ ]:
def clean_for_empty_check(text):

    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove mentions
    text = re.sub(r"@\w+", "", text)

    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # Remove whitespace
    text = re.sub(r"\s+", "", text)

    return text

In [ ]:
cleaned_text = (
    CrisisLex26[text_col]
    .fillna("")
    .apply(clean_for_empty_check)
)

empty_after_cleaning = cleaned_text.eq("")

print(
    "Tweets empty after cleaning:",
    empty_after_cleaning.sum()
)

print(
    "Percentage:",
    round(
        empty_after_cleaning.mean() * 100,
        2
    ),
    "%"
)

**Check pure URLs**

In [ ]:
pure_url = (
    CrisisLex26[text_col]
    .fillna("")
    .astype(str)
    .str.match(
        r"^\s*(https?://\S+|www\.\S+)\s*$",
        case=False
    )
)

print("Pure URL tweets:", pure_url.sum())

**Check pure @mention**

In [ ]:
pure_mention = (
    CrisisLex26[text_col]
    .fillna("")
    .astype(str)
    .str.match(
        r"^\s*(@\w+\s*)+$"
    )
)

print("Pure @mention tweets:", pure_mention.sum())

**Check HTML entities**

In [ ]:
html_entity_mask = (
    CrisisLex26[text_col]
    .fillna("")
    .astype(str)
    .str.contains(
        r"&amp;|&lt;|&gt;|&quot;|&#\d+;",
        regex=True,
        case=False
    )
)

print(
    "Tweets containing HTML entities:",
    html_entity_mask.sum()
)

In [ ]:
CrisisLex26.loc[
    html_entity_mask,
    [text_col]
].head(20)

**Check whether HTML unescaping changes anything**

In [ ]:
html_changed = (
    CrisisLex26[text_col]
    .fillna("")
    .astype(str)
    .apply(
        lambda x: html.unescape(x) != x
    )
)

print(
    "Tweets affected by HTML unescaping:",
    html_changed.sum()
)

**Check apostrophe/contraction issues**

In [ ]:
contraction_patterns = [
    "youre",
    "dont",
    "cant",
    "wont",
    "isnt",
    "didnt",
    "wasnt",
    "theyre"
]

for word in contraction_patterns:

    count = (
        CrisisLex26[text_col]
        .fillna("")
        .astype(str)
        .str.contains(
            rf"\b{word}\b",
            case=False,
            regex=True
        )
        .sum()
    )

    print(f"{word}: {count}")

**Tweet character length**

In [ ]:
tweet_length = (
    CrisisLex26[text_col]
    .fillna("")
    .astype(str)
    .str.len()
)

print("Minimum:", tweet_length.min())
print("Maximum:", tweet_length.max())
print("Mean:", tweet_length.mean())
print("50th percentile:", tweet_length.quantile(0.50))
print("95th percentile:", tweet_length.quantile(0.95))
print("99th percentile:", tweet_length.quantile(0.99))

**token count**

In [ ]:
def count_tokens(text):

    return len(
        re.findall(
            r"\b\w+\b",
            str(text)
        )
    )


token_counts = (
    CrisisLex26[text_col]
    .fillna("")
    .apply(count_tokens)
)

print("Minimum:", token_counts.min())
print("Maximum:", token_counts.max())
print("Mean:", token_counts.mean())
print("50th percentile:", token_counts.quantile(0.50))
print("95th percentile:", token_counts.quantile(0.95))
print("99th percentile:", token_counts.quantile(0.99))

**Vacabulary size**

In [ ]:
vocabulary = set()

for text in CrisisLex26[text_col].fillna("").astype(str):

    words = re.findall(
        r"\b[a-zA-Z]+\b",
        text.lower()
    )

    vocabulary.update(words)

print("Vocabulary size:", len(vocabulary))

**Hashtag frequency**

In [ ]:
all_hashtags = []

for text in CrisisLex26[text_col].fillna("").astype(str):

    hashtags = re.findall(
        r"#\w+",
        text.lower()
    )

    all_hashtags.extend(hashtags)

hashtag_counter = Counter(all_hashtags)

print("Total hashtag occurrences:",
      len(all_hashtags))

print("Unique hashtags:",
      len(hashtag_counter))

In [ ]:
top_hashtags = pd.DataFrame(
    hashtag_counter.most_common(20),
    columns=["Hashtag", "Frequency"]
)

display(top_hashtags)

**Check hashtag dominance**

In [ ]:
if hashtag_counter:

    top_hashtag, top_count = (
        hashtag_counter.most_common(1)[0]
    )

    total_hashtags = sum(
        hashtag_counter.values()
    )

    percentage = (
        top_count / total_hashtags * 100
    )

    print("Top hashtag:", top_hashtag)
    print("Frequency:", top_count)
    print(
        "Share of all hashtag occurrences:",
        round(percentage, 2),
        "%"
    )

**@Mention frequency**

In [ ]:
mentions_per_tweet = (
    CrisisLex26[text_col]
    .fillna("")
    .astype(str)
    .apply(
        lambda x: len(
            re.findall(r"@\w+", x)
        )
    )
)

print(
    "Total mentions:",
    mentions_per_tweet.sum()
)

print(
    "Tweets containing mentions:",
    (mentions_per_tweet > 0).sum()
)

print(
    "Average mentions per tweet:",
    mentions_per_tweet.mean()
)

print(
    "Maximum mentions in one tweet:",
    mentions_per_tweet.max()
)

**URL frequency**

In [ ]:
urls_per_tweet = (
    CrisisLex26[text_col]
    .fillna("")
    .astype(str)
    .apply(
        lambda x: len(
            re.findall(
                r"https?://\S+|www\.\S+",
                x
            )
        )
    )
)

print(
    "Total URLs:",
    urls_per_tweet.sum()
)

print(
    "Tweets containing URLs:",
    (urls_per_tweet > 0).sum()
)

print(
    "Average URLs per tweet:",
    urls_per_tweet.mean()
)

print(
    "Maximum URLs in one tweet:",
    urls_per_tweet.max()
)

**Emoji frequency and diversity**

In [ ]:
def extract_emojis(text):

    emojis = []

    for char in str(text):

        code = ord(char)

        if (
            0x1F300 <= code <= 0x1FAFF
            or 0x2600 <= code <= 0x27BF
        ):
            emojis.append(char)

    return emojis

In [ ]:
all_emojis = []

for text in CrisisLex26[text_col].fillna("").astype(str):

    all_emojis.extend(
        extract_emojis(text)
    )

emoji_counter = Counter(all_emojis)

print(
    "Total emoji occurrences:",
    len(all_emojis)
)

print(
    "Unique emojis:",
    len(emoji_counter)
)

print(
    "Tweets containing emojis:",
    sum(
        len(extract_emojis(text)) > 0
        for text in CrisisLex26[text_col].fillna("")
    )
)

In [ ]:
display(
    pd.DataFrame(
        emoji_counter.most_common(20),
        columns=["Emoji", "Frequency"]
    )
)

**Tweet ID uniqueness**

In [ ]:
id_col = "Tweet ID"

print("Total rows:", len(CrisisLex26))
print("Unique Tweet IDs:", CrisisLex26[id_col].nunique())
print(
    "Duplicate Tweet IDs:",
    CrisisLex26[id_col].duplicated().sum()
)

**CRISISLEX-SPECIFIC CHECKS**

**Check the three label dimensions**

In [ ]:
label_columns = [
    "Informativeness",
    "Information Type",
    "Information Source"
]

for col in label_columns:

    print("\n============================")
    print(col)
    print("============================")

    print(
        CrisisLex26[col]
        .value_counts(dropna=False)
    )

**Label percentages**

In [ ]:
for col in label_columns:

    print("\n============================")
    print(col)
    print("============================")

    percentages = (
        CrisisLex26[col]
        .value_counts(
            normalize=True,
            dropna=False
        )
        .mul(100)
        .round(2)
    )

    display(percentages)

**Distribution of 26 events**

In [ ]:
event_counts = (
    CrisisLex26["source_file"]
    .value_counts()
)

display(event_counts)

**Event percentages**

In [ ]:
event_percentages = (
    CrisisLex26["source_file"]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(2)
)

display(event_percentages)

**Create clean event name**

In [ ]:
CrisisLex26["event_name"] = (
    CrisisLex26["source_file"]
    .str.replace(
        "-tweets_labeled.csv",
        "",
        regex=False
    )
    .str.replace(
        ".csv",
        "",
        regex=False
    )
)

display(
    CrisisLex26[
        ["source_file", "event_name"]
    ].drop_duplicates()
)

**Check whether event column already exist**

In [ ]:
if "event" in CrisisLex26.columns:

    print("Event column exists.")

    print(
        CrisisLex26["event"]
        .value_counts()
    )

else:

    print("No event column exists.")

**Event $X$ Informativeness**

In [ ]:
pd.crosstab(
    CrisisLex26["event_name"],
    CrisisLex26["Informativeness"]
)

Event X Information Type

In [ ]:
pd.crosstab(
    CrisisLex26["event_name"],
    CrisisLex26["Information Type"]
)

**Event X Information Source**

In [ ]:
pd.crosstab(
    CrisisLex26["event_name"],
    CrisisLex26["Information Source"]
)

**Check duplicate IDs with different text**

In [ ]:
id_text_counts = (
    CrisisLex26
    .groupby(id_col)[text_col]
    .nunique()
)

problematic_ids = id_text_counts[
    id_text_counts > 1
]

print(
    "Tweet IDs linked to multiple different texts:",
    len(problematic_ids)
)

**Show problematic IDs**

In [ ]:
bad_ids = problematic_ids.index

display(
    CrisisLex26[
        CrisisLex26[id_col].isin(bad_ids)
    ][
        [
            id_col,
            text_col,
            "Informativeness",
            "Information Type",
            "Information Source"
        ]
    ].head(50)
)

In [ ]:
CrisisLex26["cleaned_text"] = CrisisLex26["Tweet Text"].apply(
    clean_text_for_comparison
)

print(CrisisLex26[["Tweet Text", "cleaned_text"]].head())

In [ ]:
CrisisLex26[
    ["Tweet Text", "cleaned_text"]
].head(10)

In [ ]:
# Save the refined CrisisLexT26 dataset

output_file = "/content/CrisisLexT26_refined.csv"

CrisisLex26.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("File saved successfully:")
print(output_file)

In [ ]:
from google.colab import files

files.download("/content/CrisisLexT26_refined.csv")

#**Cross-Dataset Comparison**

In [ ]:
import re
import html
import string
import pandas as pd
import numpy as np

In [ ]:
def clean_text_for_comparison(text):

    text = str(text)

    # HTML unescaping
    text = html.unescape(text)

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove @mentions
    text = re.sub(r"@\w+", "", text)

    # Remove RT
    text = re.sub(r"\brt\b", "", text)

    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
print("HumAID:", df.shape)
print("Kaggle:", data.shape)
print("CrisisLexT26:", CrisisLex26.shape)

**Vacabulary Overlap**

In [ ]:
import re

def get_vocabulary(series):
    vocabulary = set()

    for text in series.fillna("").astype(str):
        words = re.findall(r"\b[a-zA-Z]+\b", text.lower())
        vocabulary.update(words)

    return vocabulary


hum_vocab = get_vocabulary(df["cleaned_text"])
crisis_vocab = get_vocabulary(CrisisLex26["cleaned_text"])

print("HumAID vocabulary:", len(hum_vocab))
print("CrisisLexT26 vocabulary:", len(crisis_vocab))

In [ ]:
import matplotlib.pyplot as plt

vocab_counts = {
    "HumAID": len(hum_vocab),
    "CrisisLexT26": len(crisis_vocab)
}

plt.figure(figsize=(8, 5))
plt.bar(vocab_counts.keys(), vocab_counts.values())

plt.xlabel("Dataset")
plt.ylabel("Unique Vocabulary")
plt.title("Vocabulary Size: HumAID vs CrisisLexT26")

plt.show()


In [ ]:
overlap = hum_vocab.intersection(crisis_vocab)

coverage = (len(overlap) / len(crisis_vocab)) * 100

print("HumAID vocabulary:", len(hum_vocab))
print("CrisisLexT26 vocabulary:", len(crisis_vocab))
print("Shared vocabulary:", len(overlap))
print(f"CrisisLexT26 vocabulary covered by HumAID: {coverage:.2f}%")

**Tweet length Distribution**

*Calculate cleaned tweet length*

In [ ]:
df["comparison_length"] = (
    df["cleaned_text"].fillna("").astype(str).str.len()
)

data["comparison_length"] = (
    data["cleaned_text"].fillna("").astype(str).str.len()
)

CrisisLex26["comparison_length"] = (
    CrisisLex26["cleaned_text"].fillna("").astype(str).str.len()
)

print("Length calculation completed.")

*Overlaid Histogram*

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.hist(
    df["comparison_length"],
    bins=50,
    alpha=0.4,
    label="HumAID"
)

plt.hist(
    data["comparison_length"],
    bins=50,
    alpha=0.4,
    label="Kaggle"
)

plt.hist(
    CrisisLex26["comparison_length"],
    bins=50,
    alpha=0.4,
    label="CrisisLexT26"
)

plt.xlabel("Tweet Length (Characters)")
plt.ylabel("Frequency")
plt.title("Tweet Length Distribution Across Three Datasets")
plt.legend()

plt.show()

*Numerical comparison*

In [ ]:
length_summary = pd.DataFrame({
    "HumAID": df["comparison_length"].describe(),
    "Kaggle": data["comparison_length"].describe(),
    "CrisisLexT26": CrisisLex26["comparison_length"].describe()
})

display(length_summary)

**Label Taxonomy Feasibility**

In [ ]:
print("HumAID Classes:")
print(df["class_label"].value_counts())

In [ ]:
print("CrisisLexT26 Information Type Categories:")
print(CrisisLex26["Information Type"].value_counts())

In [ ]:
taxonomy_comparison = pd.DataFrame({
    "HumAID Classes": pd.Series(df["class_label"].value_counts()),
    "CrisisLexT26 Information Type": pd.Series(
        CrisisLex26["Information Type"].value_counts()
    )
})

display(taxonomy_comparison)

In [ ]:
taxonomy_mapping = pd.DataFrame({
    "CrisisLexT26 Information Type": [
        "Affected individuals",
        "Caution and advice",
        "Donations and volunteering",
        "Infrastructure and utilities",
        "Not applicable",
        "Not labeled",
        "Other Useful Information",
        "Sympathy and support"
    ],
    "Mapped HumAID Class": [
        "injured_or_dead_people",
        "caution_and_advice",
        "rescue_volunteering_or_donation_effort",
        "infrastructure_and_utility_damage",
        "not_humanitarian",
        "No direct mapping",
        "other_relevant_information",
        "sympathy_and_support"
    ],
    "Mapping Quality": [
        "Rough",
        "Good",
        "Good",
        "Good",
        "Rough",
        "No mapping",
        "Rough",
        "Good"
    ]
})

display(taxonomy_mapping)

*Document unmapped categories*

In [ ]:
unmapped_categories = taxonomy_mapping[
    taxonomy_mapping["Mapping Quality"] == "No mapping"
]

print("CrisisLexT26 categories with no clean HumAID mapping:")
display(unmapped_categories)

**Preprossing Equivalence**

*Side-by-side spot check*

In [ ]:
spot_check = pd.DataFrame({
    "HumAID": df["cleaned_text"].head(5).reset_index(drop=True),
    "Kaggle": data["cleaned_text"].head(5).reset_index(drop=True),
    "CrisisLexT26": CrisisLex26["cleaned_text"].head(5).reset_index(drop=True)
})

display(spot_check)

*Check whether cleaning produced empty text*

In [ ]:
print("Empty cleaned texts:")

print(
    "HumAID:",
    (df["cleaned_text"].fillna("").str.strip() == "").sum()
)

print(
    "Kaggle:",
    (data["cleaned_text"].fillna("").str.strip() == "").sum()
)

print(
    "CrisisLexT26:",
    (CrisisLex26["cleaned_text"].fillna("").str.strip() == "").sum()
)

*Compare basic Preprossing output*

In [ ]:
preprocessing_summary = pd.DataFrame({
    "Dataset": ["HumAID", "Kaggle", "CrisisLexT26"],
    "Total Rows": [
        len(df),
        len(data),
        len(CrisisLex26)
    ],
    "Empty Cleaned Text": [
        (df["cleaned_text"].fillna("").str.strip() == "").sum(),
        (data["cleaned_text"].fillna("").str.strip() == "").sum(),
        (CrisisLex26["cleaned_text"].fillna("").str.strip() == "").sum()
    ]
})

display(preprocessing_summary)